NOTICE: Code is optimized for the A100 GPU

## 1. Installing and importing libraries

In [ ]:
!pip install uv

In [ ]:
!uv pip install torch numpy matplotlib lightning flash-linear-attention huggingface_hub triton pandas wandb optuna optuna-integration[pytorch_lightning]

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb

import lightning as L
from datasets import load_dataset
from fla.layers import GatedLinearAttention, LinearAttention
from huggingface_hub import HfApi, hf_hub_download, login
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import optuna
from optuna.integration import PyTorchLightningPruningCallback

## 2. Models developed

In [ ]:
# chnage code here

## 3. Data and running each model in its benchmark

Data

In [ ]:
repo_id = "monteirot/lra-benchmarks"

files = ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt', 'cifar10_sequential_lra.pt', 'pathfinder_lra.pt', 'pathx_lra.pt']

files = ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt', 'cifar10_sequential_lra.pt', 'pathfinder_lra.pt']

files = ['pathfinder_lra.pt']


for f in files:
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", local_dir=".")

In [ ]:
def make_trainer():
    return L.Trainer(
        max_epochs=6,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        logger=WandbLogger(project="lra-benchmarks", log_model='all'),
    )

### 4.1 ListOpsDataset

In [ ]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_listops = torch.load('listops.pt', weights_only=False)

#### 4.1.1 Hyperparameters optimization use of the dataset

#### 4.1.1.1 Finding the bets hyperparameters

In [ ]:
def objective(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256, 512])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8, 16])
    lr = trial.suggest_float("lr", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    model = minimal_LSTM_with_linear_attention_clf(
        vocab_size=data_listops['vocab_size'],
        hidden_size=hidden_size,
        num_classes=num_classes,
        num_heads=num_heads,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = L.Trainer(
        max_epochs=9,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
        callbacks=[PyTorchLightningPruningCallback(trial, monitor="val_acc")],
    )

    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [ ]:
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(), study_name="listops-hpo")

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = ListOpsDataset(data_listops['train'].data)
val_dataset = ListOpsDataset(data_listops['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_listops['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study.optimize(objective, n_trials=5)

In [ ]:
best = study.best_trial.params

print(best)

print("Best trial:", study.best_trial.params)
print("Best val_acc:", study.best_trial.value)

print(f"Best val_acc: {study.best_trial.value:.4f} | Params: {study.best_trial.params}")

In [ ]:
raise Exception("Manual stop: checking results here.")

#### 4.1.1.2 Training the model with the best hyperpaparemters

In [ ]:
# Datasets & loaders
train_dataset = ListOpsDataset(data_listops['train'].data)
val_dataset = ListOpsDataset(data_listops['val'].data)

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_listops['train'].data]
num_classes = max(train_labels) + 1

# Model
model_listops = minimal_LSTM_with_linear_attention_clf(
    vocab_size=data_listops['vocab_size'],
    hidden_size=best['hidden_size'],
    num_classes=num_classes,
    num_heads=best['num_heads'],
    lr=best['lr'],
)

In [ ]:
trainer = make_trainer()

In [ ]:
trainer.fit(model_listops, train_loader, val_loader)

In [ ]:
test_dataset = ListOpsDataset(data_listops['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
trainer.test(model_listops, test_loader)

In [ ]:
raise Exception("Manual stop: checking results here.")

#### 4.1.2 Normal use of the dataset

In [ ]:
# Datasets & loaders
train_dataset = ListOpsDataset(data_listops['train'].data)
val_dataset = ListOpsDataset(data_listops['val'].data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_listops['train'].data]
num_classes = max(train_labels) + 1

# Model
model_listops = minimal_LSTM_with_linear_attention_clf(
    vocab_size=data_listops['vocab_size'],
    hidden_size=128,
    num_classes=num_classes,
    num_heads=4,
    lr=1e-3,
)

In [ ]:
trainer = make_trainer()

In [ ]:
trainer.fit(model_listops, train_loader, val_loader)

In [ ]:
test_dataset = ListOpsDataset(data_listops['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
trainer.test(model_listops, test_loader)

### 4.2 IMDbDataset

In [ ]:
class IMDbDataset(Dataset):
    """PyTorch Dataset for byte-level IMDb classification."""
    def __init__(self, data, max_len=4096):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad to max_len
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_imdb = torch.load('imdb_lra.pt', weights_only=False)

#### 4.2.1 Hyperparameters optimization use of the dataset

4.2.1.1 Finding the bets hyperparameters

In [ ]:
def objective_imdb(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [16, 32, 64, 128, 256, 512])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8, 16, 32])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    model = minimal_LSTM_with_linear_attention_clf(
        vocab_size=data_imdb['vocab_size'],
        hidden_size=hidden_size,
        num_classes=num_classes,
        num_heads=num_heads,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = L.Trainer(
        max_epochs=12,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
        callbacks=[PyTorchLightningPruningCallback(trial, monitor="val_acc")],
    )

    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [ ]:
study_imdb = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(), study_name="imdb-hpo")

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = IMDbDataset(data_imdb['train'].data)
val_dataset = IMDbDataset(data_imdb['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_imdb['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study_imdb.optimize(objective_imdb, n_trials=5)

In [ ]:
best = study_imdb.best_trial.params

print(best)

#chnage this
model_to_find_num_parameters = minimal_LSTM_with_linear_attention_clf(
        vocab_size=data_imdb['vocab_size'],
        hidden_size=best["hidden_size"],
        num_classes=num_classes,
        num_heads=8,
        lr=0.0009500059493440261,
    )


# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_imdb.best_trial.params)
print("Best val_acc:", study_imdb.best_trial.value)

print(f"Best val_acc: {study_imdb.best_trial.value:.4f} | Params: {study_imdb.best_trial.params}")

print(f"Model Parameters: {total_params}")

In [ ]:
raise Exception("Manual stop: checking results here.")

#### 4.2.1.1 Running the best model with the best hyperparameters

In [ ]:
# Datasets & loaders
train_dataset = IMDbDataset(data_imdb['train'].data)
val_dataset = IMDbDataset(study_imdb['val'].data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_imdb['train'].data]
num_classes = max(train_labels) + 1

# Model
model_listops = minimal_LSTM_with_linear_attention_clf(
    vocab_size=data_imdb['vocab_size'],
    hidden_size=128,
    num_classes=num_classes,
    num_heads=4,
    lr=1e-3,
)

In [ ]:
trainer = make_trainer()

In [ ]:
trainer.fit(model_imdb, train_loader, val_loader)

In [ ]:
test_dataset = IMDbDataset(data_imdb['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
trainer.test(model_imdb, test_loader)

#### 4.2.2 Normal use of the dataset

In [ ]:
# Datasets & loaders
train_dataset = IMDbDataset(data_imdb['train'].data)
val_dataset = IMDbDataset(data_imdb['val'].data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_imdb['train'].data]
num_classes = max(train_labels) + 1

# Model
model_imdb = minimal_LSTM_with_linear_attention_clf(
    vocab_size=data_imdb['vocab_size'],
    hidden_size=128,
    num_classes=num_classes, # Use the calculated num_classes
    num_heads=4,
    lr=1e-3,
)

In [ ]:
trainer = make_trainer()

In [ ]:
trainer.fit(model_imdb, train_loader, val_loader)

In [ ]:
test_dataset = IMDbDataset(data_imdb['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
trainer.test(model_imdb, test_loader)

### 4.3 ACLRetricalDataset

In [ ]:
class ACLRetrievalDataset(Dataset):
    """PyTorch Dataset for ACL document retrieval."""
    def __init__(self, data, max_len=8001):  # 4000 + 1 (sep) + 4000
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_retrieval = torch.load('acl_retrieval_lra.pt', weights_only=False)

#### 4.3.1 Hyperparameters optimization use of the dataset

4.3.1.1 Finding the bets hyperparameters

In [ ]:
def objective_retrival(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [16, 32, 64, 128, 256, 512])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8, 16, 32])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    model = minimal_LSTM_with_linear_attention_clf(
        vocab_size=data_retrieval['vocab_size'],
        hidden_size=hidden_size,
        num_classes=num_classes,
        num_heads=num_heads,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = L.Trainer(
        max_epochs=12,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
        callbacks=[PyTorchLightningPruningCallback(trial, monitor="val_acc")],
    )

    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [ ]:
study_retrival = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(), study_name="retrival-hpo")

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = ACLRetrievalDataset(data_retrieval['train'].data)
val_dataset = ACLRetrievalDataset(data_retrieval['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_retrieval['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study_retrival.optimize(objective_retrival, n_trials=5)

In [ ]:
best = study_retrival.best_trial.params
#chnage this
model_to_find_num_parameters = minimal_LSTM_with_linear_attention_clf(
        vocab_size=data_retrieval['vocab_size'],
        hidden_size=best["hidden_size"],
        num_classes=num_classes,
        num_heads=best["num_heads"],
        lr=best["lr"],
    )

# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_retrival.best_trial.params)
print("Best val_acc:", study_retrival.best_trial.value)

print(f"Best val_acc: {study_retrival.best_trial.value:.4f} | Params: {study_retrival.best_trial.params}")

print(f"Model Parameters: {total_params}")

##### 4.3.2 Normal run

In [ ]:
# Datasets & loaders
train_dataset = ACLRetrievalDataset(data_retrieval['train'].data)
val_dataset = ACLRetrievalDataset(data_retrieval['val'].data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_retrieval['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
# Model
model_acl = minimal_LSTM_with_linear_attention_clf(
    vocab_size=data_retrieval['vocab_size'],
    hidden_size=128,
    num_classes=num_classes, # Use the calculated num_classes
    num_heads=4,
    lr=1e-3,
)

In [ ]:
trainer = make_trainer()

In [ ]:
trainer.fit(model_acl, train_loader, val_loader)

In [ ]:
test_dataset = ACLRetrievalDataset(data_retrieval['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
trainer.test(model_acl, test_loader)

### 4.4 CIFAR10SequentialDataset

In [ ]:
class CIFAR10SequentialDataset(Dataset):
    """PyTorch Dataset for sequential CIFAR-10 classification."""
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Handle both dict and tuple formats
        if isinstance(item, dict):
            tokens = item['input_ids']
            label = item['labels']
        else:
            tokens = item[0]
            label = item[-1]

        # Convert tensors to lists if needed (for padding logic)
        if isinstance(tokens, torch.Tensor):
            tokens = tokens.tolist()
        if isinstance(label, torch.Tensor):
            label = label.item()

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_cifar10_sequential = torch.load('cifar10_sequential_lra.pt', weights_only=False)

#### 4.4.1 Hyperparameters optimization use of the dataset

4.4.1.1 Finding the bets hyperparameters

In [ ]:
def objective_cifar(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [16, 32, 64, 128, 256, 512])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8, 16, 32])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    model = minimal_LSTM_with_linear_attention_clf(
        vocab_size=data_cifar10_sequential['vocab_size'],
        hidden_size=hidden_size,
        num_classes=num_classes,
        num_heads=num_heads,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = L.Trainer(
        max_epochs=12,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
        callbacks=[PyTorchLightningPruningCallback(trial, monitor="val_acc")],
    )

    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [ ]:
study_cifar = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(), study_name="imdb-hpo")

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['train'].data)
val_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_cifar10_sequential['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study_cifar.optimize(objective_cifar, n_trials=5)

In [ ]:
best = study_cifar.best_trial.params

#chnage this
model_to_find_num_parameters = minimal_LSTM_with_linear_attention_clf(
        vocab_size=data_cifar10_sequential['vocab_size'],
        hidden_size=best["hidden_size"],
        num_classes=num_classes,
        num_heads=best["num_heads"],
        lr=best["lr"],
    )


# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_cifar.best_trial.params)
print("Best val_acc:", study_cifar.best_trial.value)

print(f"Best val_acc: {study_cifar.best_trial.value:.4f} | Params: {study_cifar.best_trial.params}")

print(f"Model Parameters: {total_params}")

#### 4.4.2 Normal run

In [ ]:
# Datasets & loaders
train_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['train'])
val_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['val'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Model
model_cifar = minimal_LSTM_with_linear_attention_clf(
    vocab_size=data_cifar10_sequential['vocab_size'],
    hidden_size=128,
    num_classes=data_cifar10_sequential['num_classes'],
    num_heads=4,
    lr=1e-3,
)

In [ ]:
trainer = make_trainer()

In [ ]:
trainer.fit(model_cifar, train_loader, val_loader)

In [ ]:
test_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['test'])
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
trainer.test(model_cifar, test_loader)

## 5. Seeing results

In [ ]:
#chnage code here - list all hyperparameters runs